# Orthometric Height Calculator for GCP Feature Services

This notebook reads a Ground Control Point (GCP) feature service hosted on
ArcGIS Online (or ArcGIS Enterprise), and for every GCP that does not yet
have an orthometric height, it:

1. Queries the [NOAA NGS Geoid Height API](https://geodesy.noaa.gov/GEOID/GEOID18/computation.html)
   for the geoid separation at that point's latitude/longitude.
2. Calculates orthometric height as `ellipsoidal (GNSS) altitude - geoid separation`.
3. Writes the geoid separation and orthometric height values back to the
   feature service.
4. Saves a CSV snapshot of the updated GCP table for a local record.

Created by Ryan Lennon.

## Requirements

- Python 3.9+
- [`arcgis`](https://developers.arcgis.com/python/) (ArcGIS API for Python)
- `pandas`
- `requests`
- An authenticated ArcGIS Online / Enterprise session (see the
  **Connect to ArcGIS Online** cell below)

```bash
pip install arcgis pandas requests
```

## Before you run this

Update the values in the **Configuration** cell to match your organization's
GCP feature layer (item ID and field names).

In [ ]:
import os
from datetime import date

import pandas as pd
import requests
from arcgis.gis import GIS

## Configuration

Update these values for your organization before running the notebook.

In [ ]:
# ArcGIS Online / Enterprise item ID of the GCP feature layer to update
GCP_ITEM_ID = "YOUR_ITEM_ID_HERE"

# Index of the layer within the item (usually 0 for a single-layer service)
GCP_LAYER_INDEX = 0

# Field names on the feature layer (update to match your schema)
FIELD_OBJECTID = "OBJECTID"
FIELD_LATITUDE = "esrignss_latitude"
FIELD_LONGITUDE = "esrignss_longitude"
FIELD_ALTITUDE = "esrignss_altitude"
FIELD_GEOID_SEPARATION = "esrignss_geoidSeparation"
FIELD_ORTHOMETRIC_HEIGHT = "OrthometricHeight"

# NOAA NGS Geoid Height API endpoint
NOAA_GEOID_API_URL = "https://geodesy.noaa.gov/api/geoid/ght"

# Directory (relative to this notebook, by default) to write the output CSV log
OUTPUT_DIR = "./output"

## Connect to ArcGIS Online

`GIS("home")` works when this notebook is run inside ArcGIS Online Notebooks
or ArcGIS Pro's notebook environment, where credentials are provided
automatically. Running it elsewhere (e.g. plain Jupyter), connect explicitly,
for example:

```python
gis = GIS("https://www.arcgis.com", "your_username")   # prompts for a password
# or
gis = GIS(profile="your_saved_profile")
```

In [ ]:
gis = GIS("home")

## Helper Functions

In [ ]:
def get_geoid_data(lat, lon):
    """Query the NOAA NGS Geoid Height API for a given latitude/longitude.

    Returns the parsed JSON response as a dict, or None if the request
    failed or returned no usable data.
    """
    params = {"lat": lat, "lon": lon}
    try:
        response = requests.get(NOAA_GEOID_API_URL, params=params, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        print(f"NOAA API request failed for ({lat}, {lon}): {exc}")
        return None

    try:
        return response.json()
    except ValueError:
        print(f"NOAA API returned a non-JSON response for ({lat}, {lon}).")
        return None


def calculate_orthometric_height(altitude, geoid_separation):
    """Orthometric height = ellipsoidal (GNSS) altitude - geoid separation."""
    if altitude is None or geoid_separation is None:
        return None
    return altitude - geoid_separation

## Find GCPs Missing Orthometric Height

In [ ]:
gcp_item = gis.content.get(GCP_ITEM_ID)
gcp_layer = gcp_item.layers[GCP_LAYER_INDEX]

gcp_featureset = gcp_layer.query(where=f"{FIELD_ORTHOMETRIC_HEIGHT} IS NULL")
gcp_df = gcp_featureset.sdf

print(f"{len(gcp_df)} GCP(s) found without orthometric height.")

## Calculate and Update Orthometric Height

In [ ]:
updated_rows = []

for feature in gcp_featureset:
    objectid = feature.attributes[FIELD_OBJECTID]
    lat = feature.attributes.get(FIELD_LATITUDE)
    lon = feature.attributes.get(FIELD_LONGITUDE)
    altitude = feature.attributes.get(FIELD_ALTITUDE)

    print(f"Processing GCP #{objectid}")

    if lat is None or lon is None:
        print(f"  Missing lat/lon for GCP #{objectid}, skipping.")
        print("-----------------")
        continue

    geoid_data = get_geoid_data(lat, lon)
    if not geoid_data:
        print(f"  No data returned from NOAA API for GCP #{objectid}.")
        print("-----------------")
        continue

    geoid_height = geoid_data.get("geoidHeight")
    geoid_model = geoid_data.get("geoidModel")
    geoid_error = geoid_data.get("error")
    geoid_units = geoid_data.get("units")

    feature.attributes[FIELD_GEOID_SEPARATION] = geoid_height
    orthometric_height = calculate_orthometric_height(altitude, geoid_height)
    feature.attributes[FIELD_ORTHOMETRIC_HEIGHT] = orthometric_height

    if orthometric_height is None:
        print(f"  Altitude data not available for GCP #{objectid}; orthometric height not calculated.")
    else:
        print(f"  Geoid height: {geoid_height} {geoid_units}")
        print(f"  Orthometric height: {orthometric_height} {geoid_units}")
        print(f"  Geoid model: {geoid_model} (+/- {geoid_error} {geoid_units})")

    updated_rows.append({
        FIELD_OBJECTID: objectid,
        "geoidHeight": geoid_height,
        "geoidModel": geoid_model,
        "error": geoid_error,
        "units": geoid_units,
        FIELD_ORTHOMETRIC_HEIGHT: orthometric_height,
    })
    print("-----------------")

update_result = gcp_layer.edit_features(updates=gcp_featureset)
print(f"DOP Rover GCP feature service updated with orthometric height for {len(updated_rows)} GCP(s).")

## Save a Local CSV Log

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

formatted_date = date.today().strftime("%Y-%m-%d")
csv_path = os.path.join(OUTPUT_DIR, f"OrthometricHeight_{formatted_date}.csv")

pd.DataFrame(updated_rows).to_csv(csv_path, index=False)
print(f"GCP update log written to {csv_path}")